# Imbalanced data: five ways to handle it
A synthetic dataset where class 0 is rare. We train logistic regression on it as it is, then try random undersampling,
random oversampling and SMOTE (imbalanced-learn, plus SMOTE written by hand once to see how it works), a balanced
random forest, class weights, and a custom loss in XGBoost. Finally, resampling inside cross-validation with an
imbalanced-learn pipeline.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer
import xgboost as xgb
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import Pipeline

## 1. The data: 400 rows, two inputs, class 0 is rare

In [2]:
# make_classification invents a dataset; weights=[0.07] puts about 7% of the rows in class 0
X, y = make_classification(n_samples=400, n_features=2, n_informative=2, n_redundant=0,
                           n_clusters_per_class=2, weights=[0.07], class_sep=0.8, flip_y=0, random_state=59)
# 80% to train, 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("training rows:", X_train.shape, " test rows:", X_test.shape)
print("training counts per class:", np.bincount(y_train), " test counts:", np.bincount(y_test))

training rows: (320, 2)  test rows: (80, 2)
training counts per class: [ 21 299]  test counts: [ 7 73]


## 2. The problem: high accuracy, the rare class ignored

In [3]:
# A 'dumb' model: always predict the most frequent class
dumb = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("dumb model accuracy:", accuracy_score(y_test, dumb.predict(X_test)))

dumb model accuracy: 0.9125


In [4]:
# One helper to score any model the same way: accuracy, then precision, recall and F1 for the rare class 0, and ROC AUC
def scores(name, y_pred, y_prob=None):
    row = {"model": name,
           "accuracy": accuracy_score(y_test, y_pred),
           "precision (0)": precision_score(y_test, y_pred, pos_label=0, zero_division=0),
           "recall (0)": recall_score(y_test, y_pred, pos_label=0),
           "F1 (0)": f1_score(y_test, y_pred, pos_label=0)}
    if y_prob is not None:
        row["ROC AUC"] = roc_auc_score(y_test, y_prob)
    return row

results = []
base = LogisticRegression().fit(X_train, y_train)
print(classification_report(y_test, base.predict(X_test)))
results.append(scores("logistic regression, as is", base.predict(X_test), base.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       1.00      0.14      0.25         7
           1       0.92      1.00      0.96        73

    accuracy                           0.93        80
   macro avg       0.96      0.57      0.61        80
weighted avg       0.93      0.93      0.90        80



## 3. Random undersampling: drop majority rows until the classes are equal

In [5]:
# keep all 21 minority rows and 21 majority rows drawn at random (without replacement)
X_under, y_under = RandomUnderSampler(random_state=42).fit_resample(X_train, y_train)
print(X_train.shape, "->", X_under.shape, " counts:", np.bincount(y_under))

(320, 2) -> (42, 2)  counts: [21 21]


In [6]:
under = LogisticRegression().fit(X_under, y_under)
results.append(scores("random undersampling", under.predict(X_test), under.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.25,0.906
1,random undersampling,0.825,0.333,1.000,0.50,0.924


## 4. Random oversampling: copy minority rows until the classes are equal

In [7]:
# draw minority rows WITH replacement until there are 299 of them, so rows repeat
X_over, y_over = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)
print(X_train.shape, "->", X_over.shape, " counts:", np.bincount(y_over))
# how often each of the 21 minority rows now appears
_, times = np.unique(X_over[y_over == 0], axis=0, return_counts=True)
print("copies per minority row: min", times.min(), " max", times.max())

(320, 2) -> (598, 2)  counts: [299 299]
copies per minority row: min 9  max 21


In [8]:
over = LogisticRegression().fit(X_over, y_over)
results.append(scores("random oversampling", over.predict(X_test), over.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
1,random undersampling,0.825,0.333,1.000,0.500,0.924
2,random oversampling,0.825,0.316,0.857,0.462,0.918


## 5. SMOTE
First the formula on the worked example: sample (1, 1), neighbour (2, 2), random factor 0.5.

In [9]:
x, neighbour, lam = np.array([1, 1]), np.array([2, 2]), 0.5
print("new point:", x + lam * (neighbour - x))     # (1.5, 1.5), halfway

new point: [1.5 1.5]


In [10]:
def smote(X, y, k=5, seed=42):
    rng = np.random.default_rng(seed)
    mino = X[y == 0]
    # step 1: the k nearest minority neighbours of every minority point (k + 1 because each point is its own nearest)
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(mino).kneighbors(mino, return_distance=False)[:, 1:]
    n_new = (y == 1).sum() - len(mino)                # how many new points make the classes equal
    i = rng.integers(len(mino), size=n_new)           # step 2: a random minority point, n_new times
    j = nbrs[i, rng.integers(k, size=n_new)]          # step 3: one of its k neighbours, at random
    lam = rng.random((n_new, 1))                      # step 4: a random factor between 0 and 1
    new = mino[i] + lam * (mino[j] - mino[i])         # step 5: a point on the segment between them
    return np.vstack([X, new]), np.r_[y, np.zeros(n_new, int)]

X_hand, y_hand = smote(X_train, y_train)
print("by hand:", X_train.shape, "->", X_hand.shape, " counts:", np.bincount(y_hand))

by hand: (320, 2) -> (598, 2)  counts: [299 299]


In [11]:
# the same with imbalanced-learn: original rows first, then the 278 synthetic ones
X_smote, y_smote = SMOTE(k_neighbors=5, random_state=42).fit_resample(X_train, y_train)
print("imblearn:", X_train.shape, "->", X_smote.shape, " counts:", np.bincount(y_smote))
# check: every synthetic point lies on a segment between two minority points (here: inside their bounding range)
new = X_smote[len(X_train):]
mino = X_train[y_train == 0]
print("synthetic points inside the minority's range:", bool(((new >= mino.min(0)) & (new <= mino.max(0))).all()))

imblearn: (320, 2) -> (598, 2)  counts: [299 299]
synthetic points inside the minority's range: True


In [12]:
sm = LogisticRegression().fit(X_smote, y_smote)
results.append(scores("SMOTE", sm.predict(X_test), sm.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
1,random undersampling,0.825,0.333,1.000,0.500,0.924
2,random oversampling,0.825,0.316,0.857,0.462,0.918
3,SMOTE,0.825,0.294,0.714,0.417,0.910


## 6. A balanced random forest
Each tree is trained on a balanced sample: 21 rows of each class, drawn with replacement; the trees vote.

In [13]:
brf = BalancedRandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
# the defaults: every class is undersampled to 21 rows, drawn with replacement, for each tree
print({k: brf.get_params()[k] for k in ["sampling_strategy", "replacement", "bootstrap"]})
results.append(scores("balanced random forest", brf.predict(X_test), brf.predict_proba(X_test)[:, 1]))

rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
results.append(scores("random forest, as is", rf.predict(X_test), rf.predict_proba(X_test)[:, 1]))
pd.DataFrame(results).round(3)

{'sampling_strategy': 'all', 'replacement': True, 'bootstrap': False}


,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
1,random undersampling,0.825,0.333,1.000,0.500,0.924
2,random oversampling,0.825,0.316,0.857,0.462,0.918
3,SMOTE,0.825,0.294,0.714,0.417,0.910
4,balanced random forest,0.812,0.278,0.714,0.400,0.911
5,"random forest, as is",0.912,0.500,0.429,0.462,0.900


## 7. Cost-sensitive learning (a): class weights

In [14]:
# class 0 mistakes count 5, 25, then 50 times as much as class 1 mistakes
for w in [5, 25, 50]:
    cw = LogisticRegression(class_weight={0: w, 1: 1}).fit(X_train, y_train)
    results.append(scores(f"class_weight {{0: {w}, 1: 1}}", cw.predict(X_test), cw.predict_proba(X_test)[:, 1]))
cw = LogisticRegression(class_weight="balanced").fit(X_train, y_train)
print("weights used by 'balanced':", np.round(len(y_train) / (2 * np.bincount(y_train)), 2))
pd.DataFrame(results).round(3)

weights used by 'balanced': [7.62 0.54]


,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
1,random undersampling,0.825,0.333,1.000,0.500,0.924
2,random oversampling,0.825,0.316,0.857,0.462,0.918
3,SMOTE,0.825,0.294,0.714,0.417,0.910
4,balanced random forest,0.812,0.278,0.714,0.400,0.911
5,"random forest, as is",0.912,0.500,0.429,0.462,0.900
6,"class_weight {0: 5, 1: 1}",0.900,0.444,0.571,0.500,0.910
7,"class_weight {0: 25, 1: 1}",0.788,0.292,1.000,0.452,0.918
8,"class_weight {0: 50, 1: 1}",0.700,0.226,1.000,0.368,0.928


Move the slider: the heavier class 0's mistakes, the further the line moves into the majority class.

In [15]:
gx = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 50)
fig = go.Figure()
for label, colour, name in [(1, "#4C78A8", "class 1 (majority)"), (0, "#E45756", "class 0 (minority)")]:
    m = y_train == label
    fig.add_trace(go.Scatter(x=X_train[m, 0], y=X_train[m, 1], mode="markers", name=name,
                             marker=dict(color=colour, size=8, opacity=0.7)))
weights = [1, 2, 5, 10, 25, 50, 100]
for w in weights:
    m = LogisticRegression(class_weight={0: w, 1: 1}).fit(X_train, y_train)
    (a, b), c = m.coef_[0], m.intercept_[0]
    # the boundary is where a*x1 + b*x2 + c = 0, so x2 = -(a*x1 + c) / b
    fig.add_trace(go.Scatter(x=gx, y=-(a * gx + c) / b, mode="lines", name=f"boundary, weight {w}",
                             line=dict(color="black", width=3), visible=(w == 1)))
steps = [dict(method="update", label=str(w), args=[{"visible": [True, True] + [v == w for v in weights]}])
         for w in weights]
fig.update_layout(sliders=[dict(steps=steps, currentvalue=dict(prefix="class 0 weight: "))],
                  yaxis=dict(range=[X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]), template="simple_white",
                  font=dict(family="Latin Modern Roman"), height=550)
fig

## 7. Cost-sensitive learning (b): a custom loss in XGBoost
A weighted log loss: a mistake on class 0 costs `b`, a mistake on class 1 costs `a`. XGBoost needs its gradient and
Hessian (first and second derivative) with respect to the raw score `z`, for every row.

In [16]:
def weighted_logloss(a, b):
    def objective(z, dtrain):
        y = dtrain.get_label()
        p = 1 / (1 + np.exp(-z))                       # sigmoid of the raw score
        grad = b * (1 - y) * p - a * y * (1 - p)
        hess = p * (1 - p) * (a * y + b * (1 - y))
        return grad, hess
    return objective

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)
for b in [1, 3.5]:
    booster = xgb.train({"max_depth": 3, "eta": 0.1, "base_score": 0.5}, dtrain,
                        num_boost_round=100, obj=weighted_logloss(a=1, b=b))
    # with a custom objective, predict returns raw scores: ask for them and apply the sigmoid ourselves
    p = 1 / (1 + np.exp(-booster.predict(dtest, output_margin=True)))
    results.append(scores(f"XGBoost, custom loss b = {b}", (p > 0.5).astype(int), p))
pd.DataFrame(results).round(3)

,model,accuracy,precision (0),recall (0),F1 (0),ROC AUC
0,"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
1,random undersampling,0.825,0.333,1.000,0.500,0.924
2,random oversampling,0.825,0.316,0.857,0.462,0.918
3,SMOTE,0.825,0.294,0.714,0.417,0.910
4,balanced random forest,0.812,0.278,0.714,0.400,0.911
5,"random forest, as is",0.912,0.500,0.429,0.462,0.900
6,"class_weight {0: 5, 1: 1}",0.900,0.444,0.571,0.500,0.910
7,"class_weight {0: 25, 1: 1}",0.788,0.292,1.000,0.452,0.918
8,"class_weight {0: 50, 1: 1}",0.700,0.226,1.000,0.368,0.928
9,"XGBoost, custom loss b = 1",0.912,0.500,0.286,0.364,0.896


## 8. Resampling inside cross-validation
Wrong: resample first, then cross-validate (synthetic copies of a row land in both training and validation folds).
Right: an imbalanced-learn `Pipeline`, which resamples only the training part of each fold.

In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_minority = make_scorer(f1_score, pos_label=0)          # F1 of class 0

plain = cross_val_score(LogisticRegression(), X_train, y_train, cv=cv, scoring=f1_minority).mean()
pipe = Pipeline([("smote", SMOTE(random_state=42)), ("model", LogisticRegression())])
right = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=f1_minority).mean()
wrong = cross_val_score(LogisticRegression(), X_smote, y_smote, cv=cv, scoring=f1_minority).mean()
print(f"no resampling: {plain:.3f}   SMOTE in a pipeline: {right:.3f}   SMOTE before CV (leaky): {wrong:.3f}")

no resampling: 0.200   SMOTE in a pipeline: 0.426   SMOTE before CV (leaky): 0.885


In [18]:
# Where does the leaky score come from? Same leaky folds, but score only the real rows of each validation fold
real = np.r_[np.ones(len(y_train), bool), np.zeros(len(y_smote) - len(y_train), bool)]
all_rows, real_rows = [], []
for tr, va in cv.split(X_smote, y_smote):
    pred = LogisticRegression().fit(X_smote[tr], y_smote[tr]).predict(X_smote[va])
    all_rows.append(f1_score(y_smote[va], pred, pos_label=0))
    r = real[va]
    real_rows.append(f1_score(y_smote[va][r], pred[r], pos_label=0))
print(f"leaky, all validation rows: {np.mean(all_rows):.3f}   leaky, real validation rows only: {np.mean(real_rows):.3f}")

leaky, all validation rows: 0.885   leaky, real validation rows only: 0.398


## 9. All results side by side

In [19]:
pd.DataFrame(results).set_index("model").round(3)

,accuracy,precision (0),recall (0),F1 (0),ROC AUC
model,,,,,
"logistic regression, as is",0.925,1.000,0.143,0.250,0.906
random undersampling,0.825,0.333,1.000,0.500,0.924
random oversampling,0.825,0.316,0.857,0.462,0.918
SMOTE,0.825,0.294,0.714,0.417,0.910
balanced random forest,0.812,0.278,0.714,0.400,0.911
"random forest, as is",0.912,0.500,0.429,0.462,0.900
"class_weight {0: 5, 1: 1}",0.900,0.444,0.571,0.500,0.910
"class_weight {0: 25, 1: 1}",0.788,0.292,1.000,0.452,0.918
"class_weight {0: 50, 1: 1}",0.700,0.226,1.000,0.368,0.928


## 10. Real data: the mammography dataset, repeated cross-validation
The toy test set has only 7 minority observations, too few to compare techniques. Here: 11,183 real observations, 260 in the minority (calcification = class 1), 5-fold stratified cross-validation repeated 3 times, resampling inside each training fold. Every number is the average over the 15 test folds. Data copied from imbalanced-learn's `fetch_datasets` (benchmark of Ding 2011).

In [20]:
from imblearn.over_sampling import RandomOverSampler as ROS, SMOTE as SM
from imblearn.under_sampling import RandomUnderSampler as RUS
from imblearn.ensemble import BalancedRandomForestClassifier as BRF
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import xgboost as xgb

mam = pd.read_csv("data/mammography.csv.gz")
Xm, ym = mam.drop(columns="y").to_numpy(), mam["y"].to_numpy()   # y = 1: minority (calcification)
print(Xm.shape, np.bincount(ym))

def lr_pipe(*steps, **kw):
    return ImbPipeline([("scale", StandardScaler()), *steps, ("model", LogisticRegression(max_iter=1000, **kw))])

real_models = {
    "LR, as is": lr_pipe(),
    "LR + random undersampling": lr_pipe(("r", RUS(random_state=0))),
    "LR + random oversampling": lr_pipe(("r", ROS(random_state=0))),
    "LR + SMOTE": lr_pipe(("r", SM(random_state=0))),
    "LR + class_weight='balanced'": lr_pipe(class_weight="balanced"),
    "Decision tree, as is": DecisionTreeClassifier(random_state=0),
    "Decision tree + random oversampling": ImbPipeline([("r", ROS(random_state=0)), ("model", DecisionTreeClassifier(random_state=0))]),
    "Decision tree + SMOTE": ImbPipeline([("r", SM(random_state=0)), ("model", DecisionTreeClassifier(random_state=0))]),
    "Random forest, as is": RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=1),
    "Balanced random forest": BRF(n_estimators=100, random_state=0, n_jobs=1),
}

def minority_weighted(a):
    # weighted log loss with the minority as class 1: minority mistakes cost a, majority mistakes cost 1
    def objective(z, dtrain):
        y = dtrain.get_label()
        p = 1 / (1 + np.exp(-z))
        return a * y * (p - 1) + (1 - y) * p, p * (1 - p) * (a * y + (1 - y))
    return objective

def score_row(name, y_true, p):
    y_hat = (p > 0.5).astype(int)
    return (name, precision_score(y_true, y_hat, zero_division=0), recall_score(y_true, y_hat),
            f1_score(y_true, y_hat), roc_auc_score(y_true, p))

rows = []
cv_real = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
for tr, te in cv_real.split(Xm, ym):
    for name, m in real_models.items():
        m.fit(Xm[tr], ym[tr])
        rows.append(score_row(name, ym[te], m.predict_proba(Xm[te])[:, 1]))
    for name, a in [("XGBoost, ordinary log loss", 1), ("XGBoost, custom loss (minority x10)", 10)]:
        bst = xgb.train({"max_depth": 3, "eta": 0.1, "base_score": 0.5, "nthread": 1},
                        xgb.DMatrix(Xm[tr], label=ym[tr]), num_boost_round=100, obj=minority_weighted(a))
        z = bst.predict(xgb.DMatrix(Xm[te]), output_margin=True)
        rows.append(score_row(name, ym[te], 1 / (1 + np.exp(-z))))

real = (pd.DataFrame(rows, columns=["model", "precision (1)", "recall (1)", "F1 (1)", "ROC AUC"])
          .groupby("model", sort=False).mean().round(3))
real

(11183, 6) [10923   260]


,precision (1),recall (1),F1 (1),ROC AUC
model,,,,
"LR, as is",0.790,0.403,0.532,0.920
LR + random undersampling,0.171,0.846,0.284,0.923
LR + random oversampling,0.167,0.851,0.279,0.923
LR + SMOTE,0.170,0.849,0.283,0.922
LR + class_weight='balanced',0.168,0.853,0.281,0.923
"Decision tree, as is",0.623,0.601,0.610,0.749
Decision tree + random oversampling,0.625,0.572,0.595,0.730
Decision tree + SMOTE,0.428,0.703,0.530,0.808
"Random forest, as is",0.878,0.544,0.669,0.941
